# STAT163 · Week 3 · Before the lecture: aggregation and grouping

This notebook takes about **60 minutes**. Work through it before the lecture.

The tools: how to turn many rows into one number, and how to do that once per group. How
to group by two columns at once. How to count each kind of row when one table mixes kinds.
Which numbers come out wrong with no error to warn you. How to compute a group number and
keep the rows.

**How to use it.** Run the cells one at a time, from top to bottom. Three kinds of cells:

- **Read and run.** Run the cell and read the output. Most cells are this kind.
- **Predict.** Answer the question in the `# Predict:` comment before you run the cell.
  Write your answer on a new line under it. Most ask what the output will be. The ones in
  Part 3 give you the output and ask why. The answer is under **Answer** below the cell.
  Open it after you run.
- **Try it.** Change the code the way the instruction says, then run it.

One cell stops with an error on purpose.

Nothing here is submitted or graded.

**Recommended AI mode.** Read the cell and run it first. Then ask AI to explain what you
did not follow, or why your prediction was wrong. Do not send the notebook to a chat
before you have worked through it yourself.

## Load the table

The table you profiled in the Week 2 notebook: one month of sales from an online shop,
78 015 rows, one row per line on one receipt. `Invoice` is the receipt number, `StockCode`
the product code, `Description` the product name, `Quantity` how many units, `InvoiceDate`
when, `Price` per unit, `Customer ID` who bought, `Country` where it was shipped. An
invoice number that starts with C marks a cancellation.

The three fixes from the profile come first: the date to datetime, the customer id to
`Int64`, and a revenue column.

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/stat163-2026t1/week3-pre-lecture/main/data/online_retail_2010_11.csv"
df = pd.read_csv(url)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype("Int64")
df["line_revenue"] = df["Quantity"] * df["Price"]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_revenue
0,529995,48184,DOORMAT ENGLISH ROSE,6,2010-11-01 08:56:00,7.95,16316,United Kingdom,47.7
1,529995,48187,DOORMAT NEW ENGLAND,4,2010-11-01 08:56:00,7.95,16316,United Kingdom,31.8
2,529995,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-11-01 08:56:00,6.75,16316,United Kingdom,67.5
3,529995,22708,WRAP DOLLY GIRL,25,2010-11-01 08:56:00,0.42,16316,United Kingdom,10.5
4,529995,22781,GUMBALL MAGAZINE RACK,4,2010-11-01 08:56:00,7.65,16316,United Kingdom,30.6


## Part 1 — One number from many rows

An **aggregation** turns a column into one number: a sum, a mean, a count, a minimum. You
have done it: `.sum()` on a column, and on a mask to count rows. The method goes on the
column.

In [3]:
print(df["line_revenue"].sum().round(2))
print(df["line_revenue"].mean().round(2))
print(df["Quantity"].max())

1422654.64
18.24
9360


The first number is the shop's revenue for the month. It is a net figure: some lines are
cancellations with a negative revenue, and the sum subtracts them. The second is the
average of one line. It is not the average of one receipt, because one receipt has many lines. Before you
aggregate, decide which of the two your question asks for.

`df["Quantity"].sum()` and Python's own `sum(df["Quantity"])` return the same number. The
pandas method also works on a group, which is Part 2.

Three ways to count:

In [4]:
# Predict: three numbers. Which of them are equal?
print(len(df))
print(df["Customer ID"].size)
print(df["Customer ID"].count())

78015
78015
61490


<details>
<summary>Answer</summary>

The first two are equal, 78 015: `len` and `.size` count rows. `.count()` counts the
values that are not missing, so on `Customer ID` it gives 61 490, the rows that have a
customer id. An aggregation that computes from the values skips the missing ones: `.sum()`,
`.mean()` and `.count()` all work on the values that are there. `len` and `.size` count
rows, missing or not.

</details>

### Several aggregations in one call

`agg` runs one aggregation or several, and it takes three forms. With one name,
`df["Quantity"].agg("sum")` does what `.sum()` does. With a list, one result per name:

In [5]:
df["Quantity"].agg(["sum", "mean", "median", "max"]).round(2)

,Quantity
sum,673856.00
mean,8.64
median,3.00
max,9360.00


The names are strings, and pandas knows which method each one means. The result is a
Series labelled with them.

With keyword arguments, you choose the labels:

In [6]:
df["Quantity"].agg(units="sum", per_line="mean").round(2)

,Quantity
units,673856.00
per_line,8.64


After a groupby, the labels you pick become the names of the columns.

You can pass your own function in place of a name. It receives the column and returns one
number.

In [7]:
def spread(values):
    return values.max() - values.min()

df["Quantity"].agg(["min", "max", spread])

,Quantity
min,-9000
max,9360
spread,18360


The result is labelled with the function's name. Write a function when no built-in name
does what you want. When a built-in name exists, use it: it is faster, and after a groupby the
difference grows, because pandas calls your function once for every group.

### The grain changed: many rows in, one row out

Before the aggregation, one row was one line on a receipt: that is the **grain** of the
table, what one row stands for. After it, there is one row for the whole table. An
aggregation **collapses the grain**: many rows go in, one comes out, and the rows
themselves are gone from the result.

## Part 2 — One number per group

Rows per country is `value_counts()`, and you have run it. It is an aggregation done
once per value: the rows are split by country, and each piece is counted.

Revenue per country is the same split with a sum instead of a count. In plain Python that
is a loop with a dictionary: read each row, add its revenue to the entry for its country.

In [9]:
revenue = {}
for country, amount in zip(df["Country"], df["line_revenue"]):
    revenue[country] = revenue.get(country, 0) + amount

round(revenue["Germany"], 2)

29626.91

pandas does the same in one line. `groupby("Country")` splits the table into one piece per
country, `["line_revenue"].sum()` sums each piece, and the pieces come back together as
one Series, one row per country.

In [10]:
by_country = df.groupby("Country")["line_revenue"].sum()
by_country

,line_revenue
Country,
Australia,18207.820
Austria,2873.300
Belgium,5178.280
Canada,381.770
Channel Islands,1221.430
Cyprus,397.600
Denmark,3876.360
EIRE,30474.140
Finland,414.980


This is **split, apply, combine**: split the rows by a key, apply an aggregation to each
piece, combine the results into one table. The key, `Country` here, becomes the index of
the result. Read the index of the output: it has 29 labels, one per country, and the row
labels of `df` are gone.

`value_counts()` is the same three steps with a count:

In [11]:
print(df["Country"].value_counts().head(3))
print(df.groupby("Country").size().sort_values(ascending=False).head(3))

Country
United Kingdom    72438
EIRE               1121
Germany            1003
Name: count, dtype: int64
Country
United Kingdom    72438
EIRE               1121
Germany            1003
dtype: int64


`size()` counts the rows of each piece, and `value_counts()` is that count sorted largest
first.

The loop and the one-liner give the same number for Germany. The one-liner gives all 29 at
once, and it is faster: the loop takes one step in Python for each of the 78 015 rows,
while the one-liner hands the whole column to code written in C, which pandas calls once.
The result is a Series, so you can sort it:

In [12]:
by_country.sort_values(ascending=False).head()

,line_revenue
Country,
United Kingdom,1239207.632
Netherlands,33674.910
EIRE,30474.140
Germany,29626.910
France,19497.870


Run the groupby on its own, with no aggregation after it:

In [13]:
# Predict: a table, or something else?
df.groupby("Country")

<details>
<summary>Answer</summary>

An object, printed with its type and no rows. `groupby` on its own is a description of
the split. Nothing is computed until an aggregation is asked for.

</details>

### Counting inside a group

Three counts, per group.

In [14]:
# Predict: for Australia, which of the three numbers is the number of receipts?
print(df.groupby("Country")["Invoice"].size().head(3))
print(df.groupby("Country")["Invoice"].count().head(3))
print(df.groupby("Country")["Invoice"].nunique().head(3))

Country
Australia    176
Austria      116
Belgium      228
Name: Invoice, dtype: int64
Country
Australia    176
Austria      116
Belgium      228
Name: Invoice, dtype: int64
Country
Australia     7
Austria       5
Belgium      11
Name: Invoice, dtype: int64


<details>
<summary>Answer</summary>

The third. `size()` counts rows in the group, `count()` counts rows whose `Invoice` is
not missing, and here no invoice number is missing, so the first two are the same.
`nunique()` counts the distinct invoice numbers, which is the number of receipts. In plain
Python that is `len(set(...))`, with the missing values left out. Australia has 176 rows
and 7 receipts.

The rule: **a count of rows is a count of lines, not of receipts.** When the question is
"how many receipts", count the distinct values.

</details>

`count()` and `size()` give different numbers on a column with missing values. `agg`
works on a group the way it worked on a column in Part 1:

In [15]:
df.groupby("Country")["Customer ID"].agg(["size", "count"]).sort_values("size", ascending=False).head(3)

,size,count
Country,,
United Kingdom,72438,56194
EIRE,1121,987
Germany,1003,1003


### One column per aggregation

`agg` takes a name for each result column. Each name gets two things in parentheses: the
column to aggregate, and the aggregation.

In [16]:
summary = df.groupby("Country").agg(
    lines=("Invoice", "size"),
    receipts=("Invoice", "nunique"),
    customers=("Customer ID", "nunique"),
    revenue=("line_revenue", "sum"),
)
summary.sort_values("revenue", ascending=False).head()

,lines,receipts,customers,revenue
Country,,,,
United Kingdom,72438,3372,1549,1239207.632
Netherlands,470,23,7,33674.910
EIRE,1121,54,3,30474.140
Germany,1003,63,29,29626.910
France,989,46,25,19497.870


The result is a DataFrame: one row per country, one column per aggregation, `Country` as
the index.

The same groupby with `.sum()` and no column named:

In [17]:
# Predict: what happens?
df.groupby("Country").sum()

TypeError: datetime64 type does not support sum operations

<details>
<summary>Answer</summary>

A `TypeError`. `InvoiceDate` is the column that raises it: two timestamps cannot be added
to each other. `Description` would not have raised anything. pandas joins text end to end,
so a sum of it returns one long string per country, which answers nothing. Name the columns you want:
`df.groupby("Country")[["Quantity", "line_revenue"]].sum()`. Two names in a list give a
DataFrame with two columns, the same rule as `df[["a", "b"]]`.

</details>

### Two keys: one row per combination

A list of keys groups by both at once: one row per combination that occurs in the table.

In [18]:
df["is_cancelled"] = df["Invoice"].str.startswith("C")
receipts = df.groupby(["Country", "is_cancelled"])["Invoice"].nunique()
receipts.head(6)

Country    is_cancelled
Australia  False           5
           True            2
Austria    False           5
Belgium    False           7
           True            4
Canada     False           1
Name: Invoice, dtype: int64

The index of the result has two labels per row, the country and then True or False. That
is a **MultiIndex**: one index built from two levels. Every row has both labels. The
printout shows a country once and leaves that level blank on the rows under it, to save
repeating the name. Those blanks are not missing values.

A label from the first level alone selects the rows under it. Both labels, written as a
tuple, select one row.

In [19]:
print(receipts["Germany"])
print(receipts[("Germany", False)])

is_cancelled
False    43
True     20
Name: Invoice, dtype: int64
43


`reset_index()` moves the levels back into ordinary columns and gives the table the plain
0, 1, 2 index of any other table.

In [20]:
receipts_flat = receipts.reset_index()
receipts_flat.head(6)

,Country,is_cancelled,Invoice
0,Australia,False,5
1,Australia,True,2
2,Austria,False,5
3,Belgium,False,7
4,Belgium,True,4
5,Canada,False,1


**Do that most of the time.** On a table with ordinary columns, everything you already
write works: the filters, the sorts, the column selections. A MultiIndex needs its own syntax for
each of them. Keep the MultiIndex when you want to select by level, the way `receipts["Germany"]`
does.

### A key that is not a column

The key can be any Series that has the same row labels as the table. A Series you build
from one of the table's own columns has the same row labels. Here a column of weekday names, built inside
the `groupby` call, groups the receipts by weekday:

In [21]:
df.groupby(df["InvoiceDate"].dt.day_name())["Invoice"].nunique()

,Invoice
InvoiceDate,
Friday,521
Monday,595
Sunday,390
Thursday,714
Tuesday,829
Wednesday,620


Six days, not seven: the shop took no order on a Saturday that month.

**Try it.** Change `day_name()` to `hour`, with no parentheses, and add
`.sort_values(ascending=False)` at the end to read which hour of the day has the most
receipts.

In [23]:
# Try it: replace day_name() with hour, no parentheses, then add .sort_values(ascending=False)
df.groupby(df["InvoiceDate"].dt.hour())["Invoice"].nunique()

TypeError: 'Series' object is not callable

A key built inside the `groupby` call exists only in that line. That is what you want for
a question you ask once.

When you need the same key again, store it as a column, and every line below it reads the
column.

In [24]:
df["weekday"] = df["InvoiceDate"].dt.day_name()
df.groupby("weekday")["Invoice"].nunique()

,Invoice
weekday,
Friday,521
Monday,595
Sunday,390
Thursday,714
Tuesday,829
Wednesday,620


### One column per kind of row

The same idea works with a condition in place of a part of a date. The table has three
kinds of rows. A cancellation has an invoice number that starts with C. A write-off has a price
of zero and is not a cancellation: a correction to the shop's own records, most of them
with no description. Everything else is a sale. Each kind is a condition, and a condition
stored as a column is True on the rows of that kind.

In [25]:
df["is_writeoff"] = ~df["is_cancelled"] & (df["Price"] == 0)
df["is_sale"] = ~df["is_cancelled"] & ~df["is_writeoff"]
print(df["is_sale"].sum())
print(df["is_cancelled"].sum())
print(df["is_writeoff"].sum())

76464
1194
357


Every row is exactly one of the three, so the three counts add up to 78 015. Per country,
the same counts come from one `agg`, because the sum of a True/False column is the number
of rows where it is True:

In [26]:
kinds = df.groupby("Country").agg(
    lines=("Invoice", "size"),
    sales=("is_sale", "sum"),
    cancellations=("is_cancelled", "sum"),
    writeoffs=("is_writeoff", "sum"),
)
kinds.sort_values("lines", ascending=False).head()

,lines,sales,cancellations,writeoffs
Country,,,,
United Kingdom,72438,71117,968,353
EIRE,1121,1098,22,1
Germany,1003,942,58,3
France,989,978,11,0
Netherlands,470,453,17,0


One row per country, one column per kind of row. The kinds were never split into separate
tables: the condition did the splitting and the sum did the counting.

This is a **conditional count**: a condition stored as a column, summed per group, counts
the rows of each group where it holds. Any comparison you can write is a condition, so you
choose the question the count answers. Several conditions in one `agg` are counted
together, and the table is read once.

The three above name a kind of row. These two describe how much was ordered on the line,
and which day it was ordered on:

In [27]:
df["is_single_unit"] = df["Quantity"] == 1
df["is_sunday"] = df["weekday"] == "Sunday"

df.groupby("Country").agg(
    lines=("Invoice", "size"),
    single_unit_lines=("is_single_unit", "sum"),
    sunday_lines=("is_sunday", "sum"),
).sort_values("lines", ascending=False).head()

,lines,single_unit_lines,sunday_lines
Country,,,
United Kingdom,72438,23156,11235
EIRE,1121,37,47
Germany,1003,20,157
France,989,111,99
Netherlands,470,5,0


The same columns with `max` answer a different kind of question:

In [28]:
# Predict: what does this number count?
df.groupby("Country")["is_cancelled"].max().sum()

np.int64(18)

<details>
<summary>Answer</summary>

The number of countries with at least one cancellation, 18 of 29. `max` of a True/False
column per group is True when any row of the group is True, so the result is one True or
False per country, and its sum counts the countries.

The rule: **sum a condition per group to count how many rows meet it. Take its `max` to
find out whether any row meets it.**

</details>

The second question is about presence: did this happen at least once in the group? Did
this country ever cancel an order? Did this customer ever buy at a price of zero? `any`
does the same as `max`, and says so in its name.

In [29]:
df.groupby("Country")["is_cancelled"].agg(["sum", "max", "any"]).head()

,sum,max,any
Country,,,
Australia,6,True,True
Austria,0,False,False
Belgium,7,True,True
Canada,0,False,False
Channel Islands,62,True,True


## Part 3 — Aggregations that are wrong with no error

Each cell in this part runs, prints a number, and the number answers a question you did
not ask.

### Revenue per customer

The same sum with `Customer ID` as the key.

In [30]:
per_customer = df.groupby("Customer ID")["line_revenue"].sum()
per_customer.head(3)

,line_revenue
Customer ID,
12351,300.93
12352,343.80
12356,2651.81


In [ ]:
# Predict: the month's revenue was 1 422 654.64. The per-customer totals add up to less. Which rows are left out?
print(per_customer.sum().round(2))

<details>
<summary>Answer</summary>

The 16 525 rows that have no customer id: the 78 015 rows less the 61 490 that have one.
The per-customer totals come to 1 134 879.28, about a fifth less than the month's revenue.
`groupby` drops every row whose key is missing. Those rows are in the table and in the
month's revenue, and in no customer's total.

The rule: **in pandas, a groupby drops rows whose key is missing.** Before you group,
check `isna().sum()` on the key.

That is a rule of pandas. Other tools chose differently: `GROUP BY` in SQL keeps the rows
whose key is missing and puts them in a group of their own, and so does `group_by` in R's
dplyr. The same calculation moved from one tool to another can change its totals for this
reason alone.

</details>

`dropna=False` keeps those rows as one more group, with the missing marker as its label:

In [31]:
df.groupby("Customer ID", dropna=False)["line_revenue"].sum().tail(3)

,line_revenue
Customer ID,
18283,195.35
18287,381.50
<NA>,287775.36


### An average of averages

The average receipt. First the revenue of each receipt, then the mean of those:

In [32]:
per_invoice = df.groupby("Invoice")["line_revenue"].sum()
print(per_invoice.mean().round(2))

387.75


Now the same number per country. Two keys give one total per receipt with its country next
to it, and `reset_index()` makes both of them ordinary columns. A second groupby on
`Country` then averages the receipts of each country.

In [33]:
invoice_by_country = df.groupby(["Country", "Invoice"])["line_revenue"].sum().reset_index()
mean_invoice = invoice_by_country.groupby("Country")["line_revenue"].mean()
mean_invoice.sort_values(ascending=False).head()

,line_revenue
Country,
Australia,2601.117143
Switzerland,2261.788333
Lithuania,1525.390000
Netherlands,1464.126522
Israel,1248.420000


In [ ]:
# Predict: the mean receipt is 387.75. The mean of the 29 country means is 673.84. Why is it larger?
print(mean_invoice.mean().round(2))

<details>
<summary>Answer</summary>

The mean of the means counts every country once, so Australia, with 7 receipts, counts as
much as the United Kingdom with 3 372. The mean over all receipts
counts every receipt once, and most receipts are British.

The rule: **an average of group averages is not the average.** The mean over all groups
is the revenue of all receipts divided by the number of receipts, which is what
`per_invoice.mean()` computed.

</details>

### Sorting by a rate

A rate is one count divided by another: here, cancelled lines divided by all lines.
`is_cancelled` is True or False, and the mean of a True/False column is exactly that
division. Per country, sorted:

In [34]:
cancel_rate = df.groupby("Country")["is_cancelled"].mean()
cancel_rate.sort_values(ascending=False).head()

,is_cancelled
Country,
Japan,0.625000
RSA,0.500000
Channel Islands,0.389937
USA,0.187500
Spain,0.100000


Add the counts next to the rate:

In [35]:
df.groupby("Country")["is_cancelled"].agg(["mean", "sum", "size"]).sort_values("mean", ascending=False).head()

,mean,sum,size
Country,,,
Japan,0.625000,5,8
RSA,0.500000,1,2
Channel Islands,0.389937,62,159
USA,0.187500,9,48
Spain,0.100000,9,90


Sort by a rate and the small groups come to the top. Japan's five cancellations in
eight lines is a rate of 0.625, and the United Kingdom's 968 in 72 438 lines is 0.013. A
rate says nothing without the size of the group it was computed on, so a table of rates
has the count next to it.

The same question for the rates:

In [36]:
# Predict: 0.0763 as the mean of the 29 country rates, against 0.0153 over all lines. Which trap from this part is this?
print(cancel_rate.mean().round(4))
print(df["is_cancelled"].mean().round(4))

0.0763
0.0153


<details>
<summary>Answer</summary>

The average of averages, again: every country counts once, whatever its size. The
countries with few lines and high rates raise the mean of the rates, and each of them has
only a few rows. The rate over all lines is the count of cancelled rows divided by the
count of all rows, which is five times smaller.

</details>

### Customers per day

`.dt.date` gives the calendar day of every timestamp, without the time. The distinct
customers of the month, then of each day:

In [37]:
customers_per_day = df.groupby(df["InvoiceDate"].dt.date)["Customer ID"].nunique()
print(df["Customer ID"].nunique())
print(customers_per_day.head())

1683
InvoiceDate
2010-11-01     55
2010-11-02    106
2010-11-03     88
2010-11-04    138
2010-11-05     69
Name: Customer ID, dtype: int64


In [38]:
# Predict: 1 683 customers in the month, 2 645 when the daily counts are added. Who is counted more than once?
print(customers_per_day.sum())

2645


<details>
<summary>Answer</summary>

The ones who bought on more than one day. Each of them is counted once on every day they
bought, and once in the month. Sums and counts of rows add up across groups. Distinct
counts do not, because the same value can appear in several groups.

The rule: **a distinct count for the whole is computed on the whole**, never by adding
the distinct counts of the parts.

</details>

### Which sums mean something

Every numeric column accepts `.sum()`. Not every sum means something.

In [39]:
df.groupby("Country")[["Quantity", "Price", "Customer ID"]].sum().head(3)

,Quantity,Price,Customer ID
Country,,,
Australia,12882,509.22,2184134
Austria,1261,469.16,1458388
Belgium,2690,706.19,2862691


The quantity per country is a real number: units shipped. The sum of prices is not: each
price is per unit of one product, and adding the price of a mug to the price of a lamp
gives nothing. The sum of customer ids is a number with no meaning at all.

A value that can be added across rows is **additive**: a quantity, a revenue, a count. A
price, a rate and an id are not. For a non-additive column the questions are different:
the mean price of a line, the highest price, the number of distinct ids.

### A table of the four traps

| Trap | What you see | What to do |
|---|---|---|
| Missing key | group totals are less than the whole | check the key for missing values, or `dropna=False` |
| Average of averages | the mean of group means differs from the mean | compute the mean on the whole, or sum over count |
| Sorting by a rate | the smallest groups on top | put the count next to the rate |
| Adding distinct counts | more customers than exist | count the distinct values on the whole |

## Part 4 — A group number on every row

A receipt of five lines: what share of the receipt is each line? The share needs two
numbers per line. The line's revenue is on every row. The total of its receipt comes from
an aggregation: `per_invoice`, a Series with one row per receipt. The two grains:

In [40]:
print(len(df))
print(len(per_invoice))

78015
3669


`transform` is the other way to apply an aggregation after a groupby.

In [41]:
# Predict: how many rows does the result have — 3 669, one per receipt, or 78 015?
df.groupby("Invoice")["line_revenue"].transform("sum")

,line_revenue
0,347.20
1,347.20
2,347.20
3,347.20
4,347.20
...,...
78010,473.53
78011,473.53
78012,473.53
78013,473.53


<details>
<summary>Answer</summary>

78 015, one per line. `transform` computes the aggregation per group and writes the
group's number back on every row of the group, so every line of a receipt has its
receipt's total, and the index is the index of `df`. `sum()` after a groupby collapses the
grain to one row per group; `transform("sum")` keeps the grain. Without it you would look
up each row's receipt in `per_invoice`, one row at a time, which is a loop over the rows
again. `transform` does that for every row at once.

</details>

Because the result has the index of `df`, you can assign it as a new column:

In [42]:
df["invoice_total"] = df.groupby("Invoice")["line_revenue"].transform("sum")
df["share_of_invoice"] = df["line_revenue"] / df["invoice_total"]
df[["Invoice", "Description", "line_revenue", "invoice_total", "share_of_invoice"]].head()

,Invoice,Description,line_revenue,invoice_total,share_of_invoice
0,529995,DOORMAT ENGLISH ROSE,47.7,347.2,0.137385
1,529995,DOORMAT NEW ENGLAND,31.8,347.2,0.091590
2,529995,DOORMAT FANCY FONT HOME SWEET HOME,67.5,347.2,0.194412
3,529995,WRAP DOLLY GIRL,10.5,347.2,0.030242
4,529995,GUMBALL MAGAZINE RACK,30.6,347.2,0.088134


Now a question that needs both grains at once can be answered. On receipts with several
lines, find the lines that are more than half of the receipt. `is_sale` keeps only the
sales. On a cancellation both numbers are negative, where "more than half" means nothing.
On a write-off the revenue is zero, and a receipt of write-offs divides zero by zero, which
gives a missing value. A share of 1 is a line that is the whole receipt, and the second
condition removes those.

In [43]:
big_lines = df[df["is_sale"] & (df["share_of_invoice"] > 0.5) & (df["share_of_invoice"] < 1)]
big_lines[["Invoice", "Description", "line_revenue", "invoice_total"]].head()

,Invoice,Description,line_revenue,invoice_total
370,530029,BLUE HAPPY BIRTHDAY BUNTING,232.50,436.98
1412,530088,RED STRIPE CERAMIC DRAWER KNOB,101.76,152.04
2265,530164,ADVENT CALENDAR GINGHAM SACK,237.60,449.16
2462,530223,BOX OF VINTAGE ALPHABET BLOCKS,408.00,715.24
2622,530270,POLYESTER FILLER PAD 45x45cm,3.10,5.60


Any aggregation works in `transform`. With `"min"` on the date, grouped by customer, a row
gets the first purchase of the customer it belongs to:

In [44]:
df["first_purchase"] = df.groupby("Customer ID")["InvoiceDate"].transform("min")
df[["Customer ID", "InvoiceDate", "first_purchase"]].tail()

,Customer ID,InvoiceDate,first_purchase
78010,14441,2010-11-30 19:35:00,2010-11-18 10:12:00
78011,14441,2010-11-30 19:35:00,2010-11-18 10:12:00
78012,14441,2010-11-30 19:35:00,2010-11-18 10:12:00
78013,14441,2010-11-30 19:35:00,2010-11-18 10:12:00
78014,14441,2010-11-30 19:35:00,2010-11-18 10:12:00


The last receipt of the month belongs to a customer who first bought on the 18th. The
difference between the two date columns is the time from that customer's first purchase to
the purchase on that row.

The 16 525 rows with no customer id get `NaT`, the missing marker for a timestamp.
`transform` follows the same rule as the groupby it comes from: a row with no key belongs
to no group, so it gets no number.

**Try it.** The same groupby on `Invoice` with `transform("max")`, stored as
`largest_line`. Then keep the rows where `line_revenue` equals `largest_line`: the biggest
line of each receipt. Print how many rows you get and compare it with the 3 669 receipts.

In [ ]:
# Try it: fill in the transform, then filter with ==
# df["largest_line"] = df.groupby("Invoice")["line_revenue"].transform(...)

You get 4 253 rows, and there are 3 669 receipts. On 403 of them two or more lines are
equal to the largest value, and `==` keeps every one of them. One receipt has 16 lines at
the same value.

A filter on a group maximum returns the ties with it, so "one row per group" is something
to check, never something to assume.

### Summary table or new column

| | `groupby(...)["col"].sum()` | `groupby(...)["col"].transform("sum")` |
|---|---|---|
| Rows in the result | one per group | one per row of `df` |
| Index of the result | the group keys | the index of `df` |
| Use it for | a summary table | a new column next to the old ones |

## What to check before you trust a group number

- What is one row of the result: a line, a receipt, a customer, a country?
- Did the key have missing values?
- Is the number a sum, a count, a distinct count, a mean or a rate? A sum and a count of
  rows add up across groups; a distinct count, a mean and a rate do not.
- If it is a mean or a rate, how big is the group it was computed on?

**Bring to the lecture:** one number from this notebook that you would not put in a
report without a second number next to it, and say which second number.

## Summary

- An aggregation turns a column into one number and collapses the grain. `.sum()`,
  `.mean()`, `.count()` and `.nunique()` all skip missing values.
- `agg` takes one name, a list of names, or keyword arguments that label the results, and
  a function of your own goes wherever a name goes.
- `groupby(key)` splits the rows, an aggregation applies to each piece, and the pieces
  combine into one row per group with the key as the index. `agg(name=(column,
  function))` gives several columns at once.
- Two keys give a MultiIndex, one index of two levels. `reset_index()` turns the levels
  back into columns, and that is what you want most of the time.
- `size()` counts rows, `count()` counts non-missing values, `nunique()` counts distinct
  values. Rows are lines; receipts need `nunique`.
- A conditional count stores a condition as a column and sums it per group: how many rows
  of the group meet it. `max` or `any` of that column says whether any row does. One `agg`
  takes as many conditions as you need.
- In pandas, a groupby drops rows whose key is missing. A mean of group means and a sum of
  distinct counts are both wrong. A rate needs the count next to it.
- `transform` computes the group number and keeps every row, so it becomes a column.